# Falcon — Training on Colab Pro (G4 GPU 100GB VRAM)

This notebook trains the Falcon multi-input Transformer for age & gender estimation.
Select a dataset using the dropdown below, then run all cells.

In [ ]:
# ============================================================
# DATASET SELECTION
# ============================================================
# Options: utk, imdb, lagenda, fairface, adience, agedb, cacd
DATASET_NAME = "lagenda"

# ============================================================
# MODEL CONFIG
# ============================================================
# Variants: falcon_d1_224, falcon_d1_384, falcon_d2_224, falcon_d2_384,
#           falcon_d3_224, falcon_d3_448, falcon_d4_224, falcon_d4_448,
#           falcon_d5_224, falcon_d5_448, falcon_d5_512
MODEL_NAME = "falcon_d5_512"

# ============================================================
# TRAINING HYPERPARAMETERS
# ============================================================
BATCH_SIZE = 256
EPOCHS = 100
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.05
NUM_WORKERS = 8
WITH_PERSONS = True
AGE_MODE = "distribution"
NUM_AGE_BINS = 101

# ============================================================
# DATA LOCATIONS (Google Drive mount)
# ============================================================
# After mounting Drive, place your dataset under:
#   /content/drive/MyDrive/falcon-data/{DATASET_NAME}/
#   ├── images/
#   └── annotations/

import os
DATA_ROOT = f"/content/drive/MyDrive/falcon-data/{DATASET_NAME}"
IMAGE_PATH = os.path.join(DATA_ROOT, "images")
ANNOT_PATH = os.path.join(DATA_ROOT, "annotations")
OUTPUT_DIR = f"/content/drive/MyDrive/falcon-checkpoints/{DATASET_NAME}"

# ============================================================
# PRETRAINED VOLO CHECKPOINT (optional)
# ============================================================
# Falcon extends VOLO. To load a pretrained VOLO backbone, set:
PRETRAINED_CKPT = ""  # e.g. "/content/drive/MyDrive/volo_d5_224.pth.tar"
# Leave empty to train from scratch.

---
## 1. Environment Setup

In [ ]:
import torch, psutil, platform

print(f"Python      : {platform.python_version()}")
print(f"Torch       : {torch.__version__}")
print(f"CUDA avail  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name()}")
    print(f"VRAM        : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    print(f"CUDA arch   : {torch.cuda.get_arch_list()}")
print(f"RAM         : {psutil.virtual_memory().total / 1e9:.1f} GB")
print(f"CPU cores   : {psutil.cpu_count(logical=True)}")

In [ ]:
# Install Falcon and its dependencies
!git clone https://github.com/hoangtung386/Falcon.git /content/Falcon
%cd /content/Falcon

# Install core ML deps first (CUDA-aware)
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 --quiet

# Install remaining dependencies
!pip install -r requirements.txt --quiet

# Install the package in editable mode
!pip install -e . --quiet

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os

print(f"Images dir      : {IMAGE_PATH}")
print(f"Annotations dir : {ANNOT_PATH}")
print(f"Output dir      : {OUTPUT_DIR}")

assert os.path.isdir(IMAGE_PATH), f"Images directory not found: {IMAGE_PATH}"
assert os.path.isdir(ANNOT_PATH), f"Annotations directory not found: {ANNOT_PATH}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

import pandas as pd
csv_files = [f for f in os.listdir(ANNOT_PATH) if f.endswith(".csv")]
for f in csv_files:
    df = pd.read_csv(os.path.join(ANNOT_PATH, f), nrows=5)
    print(f"\n{f} ({len(pd.read_csv(os.path.join(ANNOT_PATH, f)))} rows)")
    print(df.head(2).to_string(index=False))

---
## 2. Training

In [ ]:
from falcon.config import ModelConfig
from falcon.model.factory import create_model
from falcon.data.dataset import build as build_data
from falcon.losses import AgeGenderLoss, OrdinalAgeLoss, WeightedMSE
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import torch.nn as nn
import torch
import logging, os, time

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
_logger = logging.getLogger("train")

device = torch.device("cuda")

In [ ]:
# Determine num_classes based on age mode
if AGE_MODE == "distribution":
    num_classes = 2 + NUM_AGE_BINS
elif AGE_MODE == "ordinal":
    num_classes = NUM_AGE_BINS
else:
    num_classes = 1

# Build config
config = ModelConfig(
    with_persons_model=WITH_PERSONS,
    use_persons=WITH_PERSONS,
    num_classes=num_classes,
    device=device,
)

print(f"Model      : {MODEL_NAME}")
print(f"Num classes: {num_classes}")
print(f"In channels: {config.in_chans}")
print(f"Input size : {config.input_size}")
print(f"Age mode   : {AGE_MODE}")
print(f"With persons: {WITH_PERSONS}")

In [ ]:
dataset, loader = build_data(
    name=DATASET_NAME,
    images_path=IMAGE_PATH,
    annotations_path=ANNOT_PATH,
    split="train",
    model_config=config,
    workers=NUM_WORKERS,
    batch_size=BATCH_SIZE,
)

print(f"Dataset: {len(dataset)} samples")
print(f"Loader : {len(loader)} batches")

In [ ]:
model = create_model(
    model_name=MODEL_NAME,
    num_classes=num_classes,
    in_chans=config.in_chans,
    pretrained=False,
    checkpoint_path=PRETRAINED_CKPT if PRETRAINED_CKPT else "",
    filter_keys=["fds."] if PRETRAINED_CKPT else None,
)
model = model.to(device)

# Count parameters
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params     : {total / 1e6:.2f}M")
print(f"Trainable params : {trainable / 1e6:.2f}M")

In [ ]:
# Loss, optimizer, scheduler, scaler
if AGE_MODE == "distribution":
    criterion = AgeGenderLoss(num_age_bins=NUM_AGE_BINS)
elif AGE_MODE == "ordinal":
    criterion = OrdinalAgeLoss(num_classes=NUM_AGE_BINS)
else:
    criterion = WeightedMSE()

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = GradScaler()

print(f"Loss      : {type(criterion).__name__}")
print(f"Optimizer : AdamW (lr={LEARNING_RATE}, wd={WEIGHT_DECAY})")
print(f"Scheduler : CosineAnnealingLR (T_max={EPOCHS})")
print(f"AMP       : Enabled")

In [ ]:
def train_epoch(model, loader, criterion, optimizer, scaler, epoch):
    model.train()
    total_loss = 0.0
    n_batches = 0
    start = time.time()

    for batch_idx, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()

        with autocast():
            outputs = model(inputs)
            loss = criterion(outputs, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        n_batches += 1

        if batch_idx % 10 == 0:
            _logger.info(
                f"Epoch {epoch} [{batch_idx}/{len(loader)}] "
                f"Loss: {loss.item():.4f}  "
                f"LR: {optimizer.param_groups[0]['lr']:.2e}"
            )

    avg_loss = total_loss / n_batches
    elapsed = time.time() - start
    _logger.info(f"Epoch {epoch} avg_loss: {avg_loss:.4f}, time: {elapsed:.1f}s")
    return avg_loss

In [ ]:
print("=" * 60)
print(f"Starting training: {DATASET_NAME} / {MODEL_NAME}")
print(f"Batch size: {BATCH_SIZE} | Epochs: {EPOCHS}")
print("=" * 60)

for epoch in range(EPOCHS):
    loss = train_epoch(model, loader, criterion, optimizer, scaler, epoch)
    scheduler.step()

    if (epoch + 1) % 10 == 0:
        ckpt_path = os.path.join(OUTPUT_DIR, f"checkpoint-{epoch}.pth.tar")
        torch.save(
            {
                "epoch": epoch,
                "state_dict": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "loss": loss,
                "min_age": dataset.min_age,
                "max_age": dataset.max_age,
                "avg_age": dataset.avg_age,
                "no_gender": False,
                "with_persons_model": WITH_PERSONS,
            },
            ckpt_path,
        )
        _logger.info(f"Saved checkpoint: {ckpt_path}")

# Save final model
final_path = os.path.join(OUTPUT_DIR, "checkpoint-final.pth.tar")
torch.save(
    {
        "epoch": EPOCHS - 1,
        "state_dict": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "loss": loss,
        "min_age": dataset.min_age,
        "max_age": dataset.max_age,
        "avg_age": dataset.avg_age,
        "no_gender": False,
        "with_persons_model": WITH_PERSONS,
    },
    final_path,
)
_logger.info(f"Training complete. Final model: {final_path}")